In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 03 — Topic Stability and Reproducibility

Menjalankan model pada seed 1–10, mencocokkan topik dengan Hungarian
algorithm, lalu melaporkan mean, standard deviation, dan 95% CI.


In [ ]:
from pathlib import Path
import sys

# Sesuaikan hanya apabila folder proyek berada di lokasi lain.
PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_ROOT.exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/HMRNet_Experiments")
    if DRIVE_ROOT.exists():
        PROJECT_ROOT = DRIVE_ROOT

assert PROJECT_ROOT.exists(), (
    "Folder HMRNet_Experiments tidak ditemukan. "
    "Unggah/ekstrak proyek ke /content atau Google Drive."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
LOGS_DIR = RESULTS_DIR / "logs"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

for directory in [TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
%cd {PROJECT_ROOT}

%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
from src.preprocessing import load_feature_bundle

bundle = load_feature_bundle(FEATURE_DIR)
raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
bow_normalized = bundle["bow_normalized"]
tfidf = bundle["tfidf"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

print("Documents:", len(raw_texts))
print("Vocabulary:", len(vocabulary))
print("Semantic shape:", semantic_embeddings.shape)
print("Document shape:", document_embeddings.shape)


In [ ]:
import joblib
import pandas as pd

from src.baseline_models import fit_lda, fit_nmf
from src.hmrnet import HMRNet, HMRNetConfig
from src.stability_metrics import evaluate_stability_runs

SEEDS = list(range(1, 11))
N_TOPICS = 20
EPOCHS = 60
METRIC = "js"


In [ ]:
beta_by_model = {
    "LDA": {},
    "NMF": {},
    "HMRNet-Full": {},
}

for seed in SEEDS:
    print(f"\nSeed {seed}")
    lda = fit_lda(
        bow=bow,
        vocabulary=vocabulary,
        n_topics=N_TOPICS,
        random_state=seed,
    )
    beta_by_model["LDA"][seed] = lda.topic_word

    nmf = fit_nmf(
        tfidf=tfidf,
        vocabulary=vocabulary,
        n_topics=N_TOPICS,
        random_state=seed,
    )
    beta_by_model["NMF"][seed] = nmf.topic_word

    hmrnet = HMRNet(
        HMRNetConfig(
            n_topics=N_TOPICS,
            epochs=EPOCHS,
            random_state=seed,
        )
    ).fit(
        bow=bow,
        semantic_embeddings=semantic_embeddings,
        document_embeddings=document_embeddings,
        vocabulary=vocabulary,
        verbose=False,
    )
    beta_by_model["HMRNet-Full"][seed] = (
        hmrnet.get_topic_word_distribution()
    )


In [ ]:
pairwise_frames = []
summaries = []

for model_name, runs in beta_by_model.items():
    pairwise, summary = evaluate_stability_runs(
        runs,
        metric=METRIC,
        confidence=0.95,
    )
    pairwise["Model"] = model_name
    pairwise_frames.append(pairwise)
    summaries.append({"Model": model_name, **summary})

pairwise_results = pd.concat(pairwise_frames, ignore_index=True)
stability_summary = pd.DataFrame(summaries).sort_values(
    "Mean_TS", ascending=False
)

pairwise_results.to_csv(
    TABLES_DIR / "experiment_03_stability_pairwise.csv",
    index=False,
)
stability_summary.to_csv(
    TABLES_DIR / "experiment_03_stability_summary.csv",
    index=False,
)
stability_summary.to_excel(
    TABLES_DIR / "experiment_03_stability_summary.xlsx",
    index=False,
)
joblib.dump(
    beta_by_model,
    CHECKPOINTS_DIR / "experiment_03_beta_by_seed.joblib",
)

display(stability_summary.round(4))


In [ ]:
import matplotlib.pyplot as plt

model_order = stability_summary["Model"].tolist()
values = [
    pairwise_results.loc[
        pairwise_results["Model"] == name,
        "Topic_Stability",
    ].to_numpy()
    for name in model_order
]

plt.figure(figsize=(8, 5))
plt.boxplot(values, labels=model_order, showmeans=True)
plt.ylabel("Topic Stability")
plt.title("Topic Stability across Random Seeds")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "experiment_03_stability_boxplot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")

TABLES_DIR = PROJECT_ROOT / "results/tables"
FIGURES_DIR = PROJECT_ROOT / "results/figures"
CHECKPOINTS_DIR = PROJECT_ROOT / "results/checkpoints"

required_files = [
    TABLES_DIR / "experiment_03_stability_pairwise.csv",
    TABLES_DIR / "experiment_03_stability_summary.csv",
    CHECKPOINTS_DIR / "experiment_03_beta_by_seed.joblib",
]

for file_path in required_files:
    print(f"{file_path.name}: {file_path.exists()}")